# # Hybrid Recommendation: ALS & TF-IDF
# **Mục tiêu:** Kết hợp sở thích ngầm định của người dùng (ALS) với đặc trưng thể loại phim (TF-IDF) để cải thiện độ chính xác của gợi ý.

In [ ]:
import pandas as pd
import numpy as np
import scipy.sparse as sp
from implicit.als import AlternatingLeastSquares
from sklearn.feature_extraction.text import TfidfVectorizer
import sys
import os

sys.path.append('../../')
from src.utils import calculate_metrics


In [ ]:
DATA_PATH = "../../data/processed/processed_ratings.pkl"
df = pd.read_pickle(DATA_PATH)

user_ids = df['userId'].astype('category').cat.codes
item_ids = df['movieId'].astype('category').cat.codes
sparse_item_user = sp.csr_matrix((np.ones(len(df)), (item_ids, user_ids)))

In [ ]:
print("Training ALS model...")
model_als = AlternatingLeastSquares(factors=64, regularization=0.1, iterations=15)
model_als.fit(sparse_item_user)

In [ ]:
movie_info = df[['movieId', 'genres']].drop_duplicates().set_index('movieId')
tfidf = TfidfVectorizer(token_pattern=r'[a-zA-Z\-]+')
tfidf_matrix = tfidf.fit_transform(movie_info['genres'].str.replace('|', ' '))
print("Evaluation starting...")